# Exercise 1: Sentiment Analysis with Hugging Face

Mục tiêu:
- Dùng pre-trained model cho sentiment analysis
- Hiểu quá trình tokenization
- Interpret model outputs (logits, probabilities, labels)

In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
print('Transformers ready')

I0000 00:00:1787107743.175232  369528 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787107743.212705  369528 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/dangkien/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warning

Transformers ready


## 1. Load Pre-trained Sentiment Analysis Pipeline

In [2]:
MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
sentiment = pipeline('sentiment-analysis', model=MODEL_NAME)
print(f'Model loaded: {MODEL_NAME}')

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Model loaded: distilbert-base-uncased-finetuned-sst-2-english


## 2. Sentiment Analysis on Sample Sentences

In [3]:
sentences = [
    'I love this product, it is absolutely amazing!',
    'This is terrible, I hate it so much.',
    'It is okay, nothing special.',
    'The movie was not bad at all.',
    'Despite some flaws, I really enjoyed the experience.',
]

results = sentiment(sentences)

print(f'{"Text":<55} {"Label":<12} {"Score"}')
print('-' * 80)
for text, result in zip(sentences, results):
    print(f'{text[:52]:<55} {result["label"]:<12} {result["score"]:.4f}')

Text                                                    Label        Score
--------------------------------------------------------------------------------
I love this product, it is absolutely amazing!          POSITIVE     0.9999
This is terrible, I hate it so much.                    NEGATIVE     0.9995
It is okay, nothing special.                            NEGATIVE     0.8290
The movie was not bad at all.                           POSITIVE     0.9980
Despite some flaws, I really enjoyed the experience.    POSITIVE     0.9998


## 3. Tokenization - Step by Step

Tokenization là quá trình chuyển text thành input cho model:
- **Tokens**: Các đơn vị subword
- **Input IDs**: Số nguyên tương ứng với mỗi token trong vocabulary
- **Attention Mask**: 1 = real token, 0 = padding

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text = 'I love this product!'

# Step 1: Tokenize thành subwords
tokens = tokenizer.tokenize(text)
print(f'Original text: {text}')
print(f'Tokens: {tokens}')

# Step 2: Convert tokens to IDs
ids = tokenizer.convert_tokens_to_ids(tokens)
print(f'Token IDs: {ids}')

# Step 3: Full encoding với special tokens ([CLS], [SEP])
encoded = tokenizer(text, return_tensors='pt')
print(f'\nFull input_ids: {encoded["input_ids"][0].tolist()}')
print(f'Attention mask: {encoded["attention_mask"][0].tolist()}')

# Step 4: Decode lại
decoded = tokenizer.decode(encoded['input_ids'][0])
print(f'Decoded: {decoded}')

print('\nGiải thích:')
print(f'  [CLS] token id={tokenizer.cls_token_id} - đánh dấu đầu chuỗi')
print(f'  [SEP] token id={tokenizer.sep_token_id} - đánh dấu cuối chuỗi')

Original text: I love this product!
Tokens: ['i', 'love', 'this', 'product', '!']
Token IDs: [1045, 2293, 2023, 4031, 999]

Full input_ids: [101, 1045, 2293, 2023, 4031, 999, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1]
Decoded: [CLS] i love this product! [SEP]

Giải thích:
  [CLS] token id=101 - đánh dấu đầu chuỗi
  [SEP] token id=102 - đánh dấu cuối chuỗi


## 4. Token-by-Token Breakdown

In [5]:
text = 'I love this product!'
full_ids = tokenizer(text)['input_ids']

print(f'{"Position":<10} {"Token ID":<12} {"Token"}')
print('-' * 40)
for pos, tid in enumerate(full_ids):
    token = tokenizer.decode([tid])
    print(f'{pos:<10} {tid:<12} {repr(token)}')

Position   Token ID     Token
----------------------------------------
0          101          '[CLS]'
1          1045         'i'
2          2293         'love'
3          2023         'this'
4          4031         'product'
5          999          '!'
6          102          '[SEP]'


## 5. Model Output Interpretation

**Logits** → **Softmax** → **Probabilities** → **Label**

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

text = 'I love this product!'
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probs = torch.softmax(logits, dim=-1)
pred = torch.argmax(probs, dim=-1).item()

labels = model.config.id2label

print(f'Text: "{text}"')
print(f'\nLogits (raw scores): {logits[0].tolist()}')
print(f'\nProbabilities (after softmax):')
for i, prob in enumerate(probs[0].tolist()):
    print(f'  {labels[i]}: {prob:.4f} ({prob*100:.2f}%)')
print(f'\nPrediction: {labels[pred]} (confidence: {probs[0][pred]:.4f})')

Text: "I love this product!"

Logits (raw scores): [-4.359877109527588, 4.715571403503418]

Probabilities (after softmax):
  NEGATIVE: 0.0001 (0.01%)
  POSITIVE: 0.9999 (99.99%)

Prediction: POSITIVE (confidence: 0.9999)


## 6. Batch Analysis với Confidence Levels

In [7]:
import matplotlib.pyplot as plt

test_sentences = [
    ('Absolutely fantastic! Best purchase ever!', 'Rõ ràng POSITIVE'),
    ('Terrible quality, complete waste of money.', 'Rõ ràng NEGATIVE'),
    ('It is okay I guess.', 'Trung tính'),
    ('Not the worst, but definitely not great.', 'Hai nghĩa'),
    ('I cannot say I am disappointed.', 'Phủ định kép'),
]

texts = [t[0] for t in test_sentences]
notes = [t[1] for t in test_sentences]
preds = sentiment(texts)

print(f'{"Text":<45} {"Note":<18} {"Label":<12} {"Score"}')
print('-' * 90)
for text, note, pred in zip(texts, notes, preds):
    print(f'{text[:42]:<45} {note:<18} {pred["label"]:<12} {pred["score"]:.4f}')

Text                                          Note               Label        Score
------------------------------------------------------------------------------------------
Absolutely fantastic! Best purchase ever!     Rõ ràng POSITIVE   POSITIVE     0.9999
Terrible quality, complete waste of money.    Rõ ràng NEGATIVE   NEGATIVE     0.9998
It is okay I guess.                           Trung tính         POSITIVE     0.9998
Not the worst, but definitely not great.      Hai nghĩa          NEGATIVE     0.9988
I cannot say I am disappointed.               Phủ định kép       NEGATIVE     0.9950


## Summary

### Kết quả quan sát:
- Model pre-trained hoạt động tốt với câu rõ ràng
- Với câu mơ hồ / phủ định kép: confidence thấp hơn
- Tokenizer chia từ thành subwords (ví dụ: `playing` → `play`, `##ing`)

### Tokenization pipeline:
```
Text → Tokens (subwords) → Token IDs → Model Input
[CLS] + tokens + [SEP] + attention_mask
```

### Model output:
```
Input → Encoder → Logits → Softmax → Probabilities → argmax → Label
```